## 📘 Notebook 04 — Feature Engineering

### Now we convert the raw retail data into model-ready features.
The key principle for RetailIQ is:

Only use information that would actually be available when making a future demand prediction.

Because this is a forecasting project, we'll be especially careful about data leakage.

## Step 1 — Import Libraries

Why: We need Pandas and NumPy for feature creation.

In [4]:
import pandas as pd
import numpy as np

### Step 2 — Load Dataset

Why: Start feature engineering from the original raw dataset.

In [5]:
df = pd.read_csv("../data/raw/sales_data.csv")

## Step 3 — Convert Date

Why: We need a proper datetime column to extract time-based features.

In [6]:
df["Date"] = pd.to_datetime(df["Date"])

## Step 4 — Sort the Data

Why: Time-series features such as lag values must be created in chronological order.

In [8]:
df = df.sort_values(
    ["Store ID", "Product ID", "Date"]
).reset_index(drop = True)

## 🕒 TIME FEATURES
### Step 5 — Extract Year

Why: Demand can change across different years.

In [9]:
df["Year"] = df["Date"].dt.year

### Step 6 — Extract Month

Why: Monthly patterns can capture seasonal demand behavior.

In [11]:
df["Month"] = df["Date"].dt.month

### Step 7 — Extract Quarter

Why: Quarterly patterns can capture broader business cycles.

In [13]:
df["Quarter"] = df["Date"].dt.quarter

### Step 8 — Extract Day of Month

Why: Some retail demand patterns can vary throughout the month.

In [14]:
df["Day"] = df["Date"].dt.day

### Step 9 — Extract Day of Week

Why: Customer demand may differ between weekdays and weekends.

0 = Monday and 6 = Sunday.

In [22]:
df["DayOfWeek"] = df["Date"].dt.dayofweek

### Step 10 — Create Weekend Feature

Why: Weekend behavior can differ from weekday behavior.
**Here**:
- 0 → Weekday
- 1 → Weekend

In [23]:
df["IsWeekend"] = (
    df["DayOfWeek"] >= 5
).astype(int)

## 📅 EXISTING BUSINESS FEATURES
### Step 11 — Encode Seasonality

Why: The dataset already provides seasonal information that can help the model.

For now, we'll keep the original categorical column.

In [24]:
df["Seasonality"] = df["Seasonality"].astype(str)

### Step 12 — Create Promotion Flag

Why: Give the feature a clearer name for modeling.

In [25]:
df["Promotion_Flag"] = df["Promotion"].astype(int)

### Step 13 — Create Discount Percentage
Why: The original discount values represent percentages, so we create a decimal representation.
**For example:**
- 10 → 0.10
- 25 → 0.25

In [26]:
df["Discount_Pct"] = df["Discount"] / 100

## 💰 PRICE FEATURES
### Step 14 — Create Competitor Price Difference

Why: The difference between our price and competitor price may be more useful than the two raw prices separately.

In [27]:
df["Price_Difference"] = (
    df["Price"] - df["Competitor Pricing"]
)

### Step 15 — Create Competitor Price Ratio

Why: A ratio tells us how our price compares proportionally with competitors.

In [28]:
df["Price_Ratio"] = (
    df["Price"] / df["Competitor Pricing"]
)

## 📦 INVENTORY FEATURES
### Step 16 — Create Inventory-to-Demand Proxy

Why: We want a simple measure of how much inventory exists relative to the demand level.

In [29]:
df["Inventory_Demand_Ratio"] = (
    df["Inventory Level"] / df["Demand"]
)

⚠️ Important: This feature is for exploratory analysis only at this stage because it uses current-period Demand. We should not use it as a predictor for future demand.

We'll later remove leakage-prone features from the final forecasting dataset.

## 🔄 LAG FEATURES

Now we create one of the most important parts of RetailIQ.

A lag feature uses past demand to help predict future demand.

For example:

Yesterday's demand → Today's prediction

### Step 17 — Lag 1

Why: Yesterday's demand can provide information about today's demand.

In [30]:
df["Demand_Lag_1"] = (
    df.groupby(["Store ID", "Product ID"])["Demand"]
    .shift(1)
)

### Step 18 — Lag 7

Why: Demand from the same weekday one week earlier can capture weekly patterns.

In [31]:
df["Demand_Lag_7"] = (
    df.groupby(["Store ID", "Product ID"])["Demand"]
    .shift(7)
)

### Step 19 — Lag 14

Why: Two-week history can capture recurring demand patterns.

In [32]:
df["Demand_Lag_14"] = (
    df.groupby(["Store ID", "Product ID"])["Demand"]
    .shift(14)
)

### Step 20 — Lag 28

Why: Four-week history can capture monthly-like demand patterns.

In [33]:
df["Demand_Lag_28"] = (
    df.groupby(["Store ID", "Product ID"])["Demand"]
    .shift(28)
)

## 📊 ROLLING FEATURES

Rolling features summarize recent historical demand.

Important: We must shift demand before calculating rolling statistics so that the current day's demand does not leak into its own prediction.

### Step 21 — 7-Day Rolling Mean

Why: Captures the average demand over the previous seven observations.

In [34]:
df["Demand_Rolling_Mean_7"] = (
    df.groupby(["Store ID", "Product ID"])["Demand"]
    .transform(
        lambda x: x.shift(1).rolling(7).mean()
    )
)

## Step 22 — 14-Day Rolling Mean

Why: Provides a longer view of recent demand.

In [35]:
df["Demand_Rolling_Mean_14"] = (
    df.groupby(["Store ID", "Product ID"])["Demand"]
    .transform(
        lambda x: x.shift(1).rolling(14).mean()
    )
)

### Step 23 — 28-Day Rolling Mean

Why: Captures longer-term demand behavior.

In [36]:
df["Demand_Rolling_Mean_28"] = (
    df.groupby(["Store ID", "Product ID"])["Demand"]
    .transform(
        lambda x: x.shift(1).rolling(28).mean()
    )
)

### Step 24 — 7-Day Rolling Standard Deviation

Why: Measures how volatile recent demand has been.

In [37]:
df["Demand_Rolling_Std_7"] = (
    df.groupby(["Store ID", "Product ID"])["Demand"]
    .transform(
        lambda x: x.shift(1).rolling(7).std()
    )
)

#### Higher values mean demand has been more variable.

### 📈 DEMAND CHANGE FEATURES
Step 25 — Demand Change From Previous Day

Why: Captures whether demand is increasing or decreasing.
- we haven't created Demand_Lag_2 yet, so first create it.

In [40]:
df["Demand_Lag_2"] = (
    df.groupby(["Store ID", "Product ID"])["Demand"]
    .shift(2)
)

#### Then run:

In [41]:
df["Demand_Change_1"] = (
    df["Demand_Lag_1"] - df["Demand_Lag_2"]
)

### Step 26 — Demand Growth From Previous Week

Why: Helps identify whether demand is increasing or decreasing compared with the previous week.

In [42]:
df["Demand_Change_7"] = (
    df["Demand_Lag_7"] - df["Demand_Lag_14"]
)

### 📦 INVENTORY RISK FEATURES
### Step 27 — Inventory Gap

Why: Estimate how far current inventory is from recent expected demand.

We'll use the previous 7-day average demand.

In [43]:
df["Inventory_Gap"] = (
    df["Inventory Level"] - df["Demand_Rolling_Mean_7"]
)

### Step 28 — Inventory Coverage

Why: Estimates approximately how many days current inventory could cover based on recent demand.

In [44]:
df["Inventory_Coverage_Days"] = (
    df["Inventory Level"] /
    df["Demand_Rolling_Mean_7"]
)

Some values will initially be NaN because the first seven observations don't have enough history.

## 🔍 CHECK THE NEW FEATURES
### Step 29 — Check Dataset Shape

Why: Confirm how many columns we created.

In [45]:
df.shape

(76000, 41)

### Step 30 — View Column Names

Why: Verify all engineered features.

In [46]:
df.columns.tolist()

['Date',
 'Store ID',
 'Product ID',
 'Category',
 'Region',
 'Inventory Level',
 'Units Sold',
 'Units Ordered',
 'Price',
 'Discount',
 'Weather Condition',
 'Promotion',
 'Competitor Pricing',
 'Seasonality',
 'Epidemic',
 'Demand',
 'Year',
 'Month',
 'Quarter',
 'Day',
 'DayofWeek',
 'DayOfWeek',
 'IsWeekend',
 'Promotion_Flag',
 'Discount_Pct',
 'Price_Difference',
 'Price_Ratio',
 'Inventory_Demand_Ratio',
 'Demand_Lag_1',
 'Demand_Lag_7',
 'Demand_Lag_14',
 'Demand_Lag_28',
 'Demand_Rolling_Mean_7',
 'Demand_Rolling_Mean_14',
 'Demand_Rolling_Mean_28',
 'Demand_Rolling_Std_7',
 'Demand_Lag_2',
 'Demand_Change_1',
 'Demand_Change_7',
 'Inventory_Gap',
 'Inventory_Coverage_Days']

### Step 31 — View Sample Rows

Why: Make sure the new features look correct.

In [47]:
df.head(15)

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,...,Demand_Lag_28,Demand_Rolling_Mean_7,Demand_Rolling_Mean_14,Demand_Rolling_Mean_28,Demand_Rolling_Std_7,Demand_Lag_2,Demand_Change_1,Demand_Change_7,Inventory_Gap,Inventory_Coverage_Days
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-01-02,S001,P0001,Electronics,North,93,71,0,65.63,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022-01-03,S001,P0001,Electronics,North,274,142,229,68.55,15,...,NaN,NaN,NaN,NaN,NaN,115.0,-31.0,NaN,NaN,NaN
3,2022-01-04,S001,P0001,Electronics,North,132,42,0,61.66,10,...,NaN,NaN,NaN,NaN,NaN,84.0,48.0,NaN,NaN,NaN
4,2022-01-05,S001,P0001,Electronics,North,319,129,0,59.56,25,...,NaN,NaN,NaN,NaN,NaN,132.0,-65.0,NaN,NaN,NaN
5,2022-01-06,S001,P0001,Electronics,North,190,98,308,84.03,0,...,NaN,NaN,NaN,NaN,NaN,67.0,43.0,NaN,NaN,NaN
6,2022-01-07,S001,P0001,Electronics,North,92,92,0,66.07,10,...,NaN,NaN,NaN,NaN,NaN,110.0,36.0,NaN,NaN,NaN
7,2022-01-08,S001,P0001,Electronics,North,308,96,248,68.56,10,...,NaN,105.857143,NaN,NaN,28.103550,146.0,-59.0,NaN,202.142857,2.909582
8,2022-01-09,S001,P0001,Electronics,North,212,57,0,62.76,10,...,NaN,105.571429,NaN,NaN,28.005102,87.0,26.0,NaN,106.428571,2.008119
9,2022-01-10,S001,P0001,Electronics,North,403,88,0,68.99,5,...,NaN,106.000000,NaN,NaN,27.640550,113.0,-26.0,NaN,297.000000,3.801887


### Step 32 — Check Missing Values

Why: Lag and rolling features naturally create missing values at the beginning of each product-store time series.

In [48]:
df.isnull().sum().sort_values(ascending=False)

Demand_Rolling_Mean_28     2800
Demand_Lag_28              2800
Demand_Lag_14              1400
Demand_Rolling_Mean_14     1400
Demand_Change_7            1400
Demand_Rolling_Mean_7       700
Demand_Rolling_Std_7        700
Demand_Lag_7                700
Inventory_Gap               700
Inventory_Coverage_Days     700
Demand_Change_1             200
Demand_Lag_2                200
Demand_Lag_1                100
Price                         0
Category                      0
Product ID                    0
Store ID                      0
Date                          0
Units Ordered                 0
Units Sold                    0
Inventory Level               0
Region                        0
Discount                      0
Promotion                     0
Weather Condition             0
Discount_Pct                  0
Promotion_Flag                0
IsWeekend                     0
DayOfWeek                     0
DayofWeek                     0
Day                           0
Quarter 

### Step 33 — Check Lag Features

Why: Confirm that historical demand is being generated correctly.

In [49]:
df[
    [
        "Date",
        "Store ID",
        "Product ID",
        "Demand",
        "Demand_Lag_1",
        "Demand_Lag_7",
        "Demand_Lag_14",
        "Demand_Lag_28"
    ]
].head(40)

,Date,Store ID,Product ID,Demand,Demand_Lag_1,Demand_Lag_7,Demand_Lag_14,Demand_Lag_28
0,2022-01-01,S001,P0001,115,NaN,NaN,NaN,NaN
1,2022-01-02,S001,P0001,84,115.0,NaN,NaN,NaN
2,2022-01-03,S001,P0001,132,84.0,NaN,NaN,NaN
3,2022-01-04,S001,P0001,67,132.0,NaN,NaN,NaN
4,2022-01-05,S001,P0001,110,67.0,NaN,NaN,NaN
5,2022-01-06,S001,P0001,146,110.0,NaN,NaN,NaN
6,2022-01-07,S001,P0001,87,146.0,NaN,NaN,NaN
7,2022-01-08,S001,P0001,113,87.0,115.0,NaN,NaN
8,2022-01-09,S001,P0001,87,113.0,84.0,NaN,NaN
9,2022-01-10,S001,P0001,99,87.0,132.0,NaN,NaN


## ⚠️ LEAKAGE CHECK
### Step 34 — Identify Features That Use Current Demand

Why: We must identify features that cannot be used to predict future demand.

In [50]:
leakage_features = [
    "Demand",
    "Inventory_Demand_Ratio"
]

leakage_features

['Demand', 'Inventory_Demand_Ratio']

**Demand** is our target, **while Inventory_Demand_Ratio** directly uses current demand.

We will not use these as forecasting predictors.

## CREATE MODELING DATASET
### Step 35 — Remove Leakage Feature

Why: Remove the exploratory feature that uses current-period demand.

In [51]:
df_model = df.drop(
    columns=["Inventory_Demand_Ratio"]
).copy()

The target **Demand** remains because we need it as the prediction target.

### Step 36 — Check Modeling Dataset

Why: Verify the final feature-engineered dataset.

In [52]:
df_model.shape

(76000, 40)

### Step 37 — Final Feature List

Why: This gives us a clear record of what the model can use.

In [53]:
feature_columns = [
    "Store ID",
    "Product ID",
    "Category",
    "Region",
    "Inventory Level",
    "Units Sold",
    "Units Ordered",
    "Price",
    "Discount",
    "Weather Condition",
    "Promotion",
    "Competitor Pricing",
    "Seasonality",
    "Epidemic",
    "Year",
    "Month",
    "Quarter",
    "Day",
    "DayOfWeek",
    "IsWeekend",
    "Promotion_Flag",
    "Discount_Pct",
    "Price_Difference",
    "Price_Ratio",
    "Demand_Lag_1",
    "Demand_Lag_2",
    "Demand_Lag_7",
    "Demand_Lag_14",
    "Demand_Lag_28",
    "Demand_Rolling_Mean_7",
    "Demand_Rolling_Mean_14",
    "Demand_Rolling_Mean_28",
    "Demand_Rolling_Std_7",
    "Demand_Change_1",
    "Demand_Change_7",
    "Inventory_Gap",
    "Inventory_Coverage_Days"
]

len(feature_columns)

37

### Step 38 — Check Final Dataset

Why: Final quality check before saving the engineered data.

In [54]:
df_model[feature_columns + ["Demand"]].info()

<class 'pandas.DataFrame'>
RangeIndex: 76000 entries, 0 to 75999
Data columns (total 38 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Store ID                 76000 non-null  str    
 1   Product ID               76000 non-null  str    
 2   Category                 76000 non-null  str    
 3   Region                   76000 non-null  str    
 4   Inventory Level          76000 non-null  int64  
 5   Units Sold               76000 non-null  int64  
 6   Units Ordered            76000 non-null  int64  
 7   Price                    76000 non-null  float64
 8   Discount                 76000 non-null  int64  
 9   Weather Condition        76000 non-null  str    
 10  Promotion                76000 non-null  int64  
 11  Competitor Pricing       76000 non-null  float64
 12  Seasonality              76000 non-null  str    
 13  Epidemic                 76000 non-null  int64  
 14  Year                     76000 no

### Step 39 — Save Feature-Engineered Dataset

Why: Preserve the engineered dataset so later notebooks don't have to recreate everything.

In [55]:
df_model.to_csv(
    "../data/processed/feature_engineered_data.csv",
    index=False
)

### Step 40 — Confirm Saved Dataset

Why: Make sure the processed file was created successfully.

In [56]:
import os

os.path.exists(
    "../data/processed/feature_engineered_data.csv"
)

True

⚠️ One correction before you run it

There is an important issue with the current raw dataset structure.

Your data has 760 unique dates, but only 76,000 rows. With 5 stores × 20 products:

5 × 20 = 100 records per date
100 × 760 = 76,000

So each Store–Product combination has a complete daily series. That's excellent for creating lag features.

However, Units Sold and Units Ordered require careful treatment. If they represent values from the same day as the target Demand, they may leak information about the period being predicted.

Therefore, for our first forecasting model, we'll eventually distinguish between:

Known before prediction

Store
Product
Category
Region
Date/time features
Weather, if available in advance
Promotion
Discount
Price
Competitor price
Historical demand

and potentially lagged operational variables such as:

Units Sold_Lag_1
Units Ordered_Lag_1
Inventory_Lag_1

rather than blindly using today's values.

We'll handle that properly when we build the forecasting dataset.